# Lewisham · `bo_system` (not a web column)

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

`bo_system` is the back-office system a record was imported from. Not shown on the website,
but it marks legacy records (Brent: `legacy_record`).

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · All `bo_system` values

A `terms` aggregation on `bo_system.raw` plus a `missing` aggregation.

In [2]:
body = {"size": 0, "track_total_hits": True, "query": lewisham_filter(),
        "aggs": {"bo_system": {"terms": {"field": "bo_system.raw", "size": 50}},
                 "none": {"missing": {"field": "bo_system.raw"}}}}
data = search(body)
print("Lewisham records:", data["hits"]["total"]["value"])
print({b["key"]: b["doc_count"] for b in data["aggregations"]["bo_system"]["buckets"]}, "| none:", data["aggregations"]["none"]["doc_count"])

Lewisham records: 29021
{'CSV Data Import': 24676, 'FME Legacy Data Import': 4312, 'Portal': 33} | none: 0


### Findings · Step 1

| bo_system | Count |
|---|---|
| CSV Data Import | 24,676 |
| FME Legacy Data Import | 4,312 |
| Portal | 33 |

Every record has a value. Brent's records are mostly CSV Data Import and FME Legacy Data
Import, but `Portal` is not new: one of Brent's OPDC duplicate copies was a Portal import
(`knowledge/brent.md`, §5; dropped there). How many Portal records Brent has is not counted.

## Step 2 · The `Portal` records

Every record with `bo_system` Portal, with the fields that have shown gaps for them
(valid_date, decision_target_date, URL) and their outcome.

In [3]:
FIELDS = ["id", "lpa_app_no", "application_type", "status", "decision", "decision_date", "valid_date",
          "decision_target_date", "appeal_decision", "url_planning_app", "last_updated", "last_updated_by", "description"]
body = {"size": 100, "_source": FIELDS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter(), {"term": {"bo_system.raw": "Portal"}}]}}}
portal = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
portal["url"] = portal.pop("url_planning_app").notna()
portal["last_updated"] = portal["last_updated"].str.slice(0, 10)
portal["description"] = portal["description"].str.slice(0, 40)
print(len(portal), "records")
print("Filled per field:", portal.notna().sum().to_dict(), "| with URL:", int(portal["url"].sum()))
print()
print(portal[["application_type", "status", "decision"]].fillna("(none)").value_counts().to_string())
print()
print(portal.drop(columns=["lpa_app_no"]).to_string(index=False))

33 records
Filled per field: {'id': 33, 'lpa_app_no': 33, 'application_type': 33, 'status': 33, 'decision': 32, 'decision_date': 32, 'valid_date': 9, 'decision_target_date': 0, 'appeal_decision': 2, 'last_updated': 33, 'last_updated_by': 33, 'description': 33, 'url': 33} | with URL: 3

application_type  status                           decision 
All Other         Completed                        Approved     17
Prior Approval    Completed                        Approved      8
All Other         Withdrawn                        Withdrawn     2
                  Commenced                        Approved      1
                  Completed                        Refused       1
                  Lapsed                           Refused       1
                  Application Under Consideration  (none)        1
                  Approved                         Approved      1
Prior Approval    Commenced                        Approved      1

                   id application_type          

### Findings · Step 2

- **Real applications, mostly built:** 26 Completed and 2 Commenced (decision Approved for
  all but 1), 2 Withdrawn, 1 Lapsed, 1 Approved, 1 pending. Decided 2009–2024; 9 are Prior
  Approvals. Descriptions are ordinary (redevelopments, conversions, Prior Approvals).
- **Few dates and links:** valid_date 9 of 33, decision_target_date 0, URL 3.
- **Updated through a "Portal":** `last_updated_by` is Portal (29) or FME Excel Import (4);
  last_updated 2025-03 to 2026-10. What this portal is (the Datahub's own, or Lewisham's)
  is not confirmed.
- **One pending since 2018:** DC/18/108548, Application Under Consideration, valid
  15/08/2018, no decision. Probably out of date; not checked.
- **2 allowed appeals** (DC/17/101020, DC/17/101621), both council decision Refused.
- **For the bot:** keep them as ordinary applications. `Portal` is not a legacy marker;
  only FME Legacy Data Import is.